# AquaHealth AI — Phase 12: CV experiment runner on Colab GPU (dataset configuration v3)

One experiment = one model × one fold × one data arm, trained by `scripts/run_cv_experiment.py` on CUDA. This notebook is the only place experiments run; the local machine is for code and manifests.

**Dataset configuration v3** (MatsyaDx-BD/Mendeley excluded; `data/audit/DATASET_CONFIG_V3.md`): 3,805 clean images, frozen final test 761 (`split_v3`), development 3,044 in 10 folds (`cv_v3`). WITH-GAN uses only the official Layer 3 artefacts generated on Colab CUDA (`results/v2/gan/`, `MyDrive/AquaHealth/aquahealth_gan_layer3_a9bc3c8.tar.gz`).

Rules baked into the code: `final_test.csv` is read for ids only; validation is always `fold_XX_validation.csv`; WITH-GAN data comes from `data/gan/fold_XX/`; a COMPLETED experiment is never retrained; `--require-cuda` aborts without a GPU.

## 1. Runtime → GPU. Clone `develop`, install the pinned dependencies

In [ ]:
!nvidia-smi
import os, pathlib, subprocess
REPO_URL = 'https://github.com/kolursamith/aquahealth.git'
BRANCH = 'develop'                                            # the single development branch
REPO_DIR = pathlib.Path('/content/aquahealth')
if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only
!git log -1 --oneline
!pip install -q -r requirements/experiments.txt   # torch/torchvision/opencv/matplotlib pins; nothing else
def sh(cmd):
    """Run a gate command, streaming its output into the cell; a non-zero exit STOPS the
    cell (unlike a bare `!` line, whose failure is ignored)."""
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    assert rc == 0, f'GATE FAILED (exit {rc}): {cmd}'

## 2. CUDA, GPU, VRAM, PyTorch version (abort here without a GPU)

`gpu_smoke.py` moves a tensor to the GPU and runs one forward/backward/optimizer step; `colab_preflight.py --env-only` reports Python, torch/torchvision vs the pins, CUDA version, GPU name, VRAM and the repository digests. Exit 2 = no CUDA.

In [ ]:
sh("python scripts/gpu_smoke.py --require-cuda")
sh("python scripts/colab_preflight.py --env-only --require-cuda")
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0),
      '| VRAM GiB', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 3. Dataset: the four delivered folders on Drive → `data/raw/<key>` → local bundle (Layer 2 mechanism)

`MyDrive/AquaHealth/` holds the source deliveries as delivered. The folder → key mapping is **not guessed**: `scripts/verify_drive_delivery.py` applies the committed mapping (`src/multi_dataset.py::DATASET_SOURCES` — v3: `current_freshwater`, `kaptai`, `roboflow`, `paper_dataset`; `MatsyaDx-BD` is in `EXCLUDED_SOURCES` and is neither expected nor read), counts images per key against the audited inventory (`clean_manifest.csv`: 3,503 / 133 / 454 / 1,208) and checks that every manifest row resolves to a file. It FAILS if an active source is missing and ignores any excluded folder. `link_raw_datasets.py` then creates the `data/raw/<key>` symlinks — the runtime path-resolution layer; no manifest is touched.

Because Drive is a slow FUSE mount, `scripts/build_colab_bundle.py` copies exactly the 3,805 *included* images to the VM's local disk (`/content/aquahealth_data/aquahealth_bundle`) and `AQUAHEALTH_COLAB_DATASET_ROOT` is pointed at that bundle. Raw files on Drive are only read.

In [ ]:
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DROP = pathlib.Path('/content/drive/MyDrive/AquaHealth')       # the five delivered folders
assert DRIVE_DROP.is_dir(), f'{DRIVE_DROP} not found'
!ls -la "{DRIVE_DROP}"
# 3a. map the visible folders to the five keys and compare with the audited inventory (exit 1 = STOP)
sh(f'python scripts/verify_drive_delivery.py --source "{DRIVE_DROP}"')
# 3b. runtime path resolution only: data/raw/<key> -> Drive folders (symlinks; manifests unchanged)
sh(f'python scripts/link_raw_datasets.py --source "{DRIVE_DROP}" --force')
!ls -la data/raw
# 3c. copy exactly the included images (3,805 in v3) to local disk; section 4 hashes every copy
DATA_ROOT = pathlib.Path('/content/aquahealth_data')
BUNDLE = DATA_ROOT / 'aquahealth_bundle'
BUNDLE_TAR = pathlib.Path('/content/drive/MyDrive/AquaHealth/aquahealth_bundle_v3.tar')   # ~100 MB: the 3,805 included images
if not (BUNDLE / 'bundle.sha256').is_file():
    if BUNDLE_TAR.is_file():
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        !tar -xf "{BUNDLE_TAR}" -C "{DATA_ROOT}"        # seconds, instead of ~30 min of Drive FUSE reads
    else:
        sh(f'python scripts/build_colab_bundle.py --out "{BUNDLE}" --no-verify-hashes')   # section 4 hashes every local copy
        !tar -cf "{BUNDLE_TAR}" -C "{DATA_ROOT}" aquahealth_bundle && ls -la "{BUNDLE_TAR}"
os.environ['AQUAHEALTH_COLAB_DATASET_ROOT'] = str(BUNDLE)
sh("python scripts/colab_dataset.py info")
sh("python scripts/colab_dataset.py attach --force")   # data/raw/<key> now -> the local bundle
!ls -la data/raw

## 4. Data integrity — 3,805 images by SHA-256, manifest / split / fold digests, frozen final test

`verify --hash all` hashes every attached image against the clean manifest and checks the bundle pins (SHA-256 of `clean_manifest.csv`, `development.csv`, `final_test.csv`, `folds.csv`), counts, canonical labels, group ids and fold ids. The second cell prints the committed digests of the frozen final test (`split_v3`) and the ten folds (`cv_v3`) and confirms no final-test id is inside any fold manifest. A non-zero exit means **STOP** — never regenerate manifests on Colab.

In [ ]:
sh("python scripts/colab_dataset.py verify --hash all")
!cat data/audit/split_v3/split_manifest.sha256; echo; cat data/audit/cv_v3/folds.sha256
import csv, hashlib, pathlib
sha = lambda p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
print('final_test.csv', sha('data/audit/split_v3/final_test.csv'))
test_ids = {r['image_id'] for r in csv.DictReader(open('data/audit/split_v3/final_test.csv'))}
for k in range(1, 11):
    for part in ('train', 'validation'):
        f = f'data/audit/cv_v3/fold_{k:02d}_{part}.csv'
        ids = {r['image_id'] for r in csv.DictReader(open(f))}
        assert not (ids & test_ids), f'{f} contains final-test ids'
        print(f'fold_{k:02d}_{part}: {len(ids):5d} rows  sha256 {sha(f)[:16]}  final-test overlap 0')
print('final test ids', len(test_ids), '— present in no fold manifest')

## 5. Persist `data/gan` and `results/v2` on Drive (recommended for the 10-fold run)

The full run takes roughly an hour; with the links below a disconnect loses nothing and `run_gan_all_folds.py` resumes at the first incomplete fold.

In [ ]:
PERSIST_ON_DRIVE = True
if PERSIST_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = pathlib.Path('/content/drive/MyDrive/AquaHealth')
    for local, remote in (('results/v2', DRIVE / 'results_v2_datasetv3'), ('data/gan', DRIVE / 'gan_datasetv3')):
        remote.mkdir(parents=True, exist_ok=True)
        local = pathlib.Path(local)
        if local.exists() and not local.is_symlink():
            !rsync -a --ignore-existing "{local}/" "{remote}/"
            !rm -rf "{local}"
        if not local.is_symlink():
            local.parent.mkdir(parents=True, exist_ok=True)
            local.symlink_to(remote)
        print(local, '->', local.resolve())

## 6. (WITH-GAN only) restore the official Layer 3 GAN artefacts

The official synthetic images, generators and per-fold manifests were generated on Colab CUDA (Layer 3, VERIFIED) and persisted as `MyDrive/AquaHealth/aquahealth_gan_layer3_a9bc3c8.tar.gz`. Extract it at the repository root and verify against the committed `results/v2/gan/` records. Nothing is regenerated here.

In [ ]:
DATA_ARM_NEEDS_GAN = True   # set False when only WITHOUT-GAN experiments run in this session
if DATA_ARM_NEEDS_GAN:
    GAN_TAR = '/content/drive/MyDrive/AquaHealth/aquahealth_gan_layer3_a9bc3c8.tar.gz'
    assert pathlib.Path(GAN_TAR).is_file(), f'{GAN_TAR} missing — the official Layer 3 export'
    if not pathlib.Path('data/gan/fold_10/generator.pt').is_file():
        !tar -xzf "{GAN_TAR}" -C . --exclude='results/v2/gan/*'   # images + generators + per-fold manifests; the tracked records stay git's
    sh("python scripts/verify_gan_outputs.py --images all")

## 7. Pre-flight for the requested experiment (integrity, dataset, manifests, model I/O, CUDA)

`--model` builds the network and proves `(N, 3, 224, 224)` → `(N, 8)` finite logits on the GPU before any training.

In [ ]:
MODEL = 'cnn_vit_lstm'        # cnn_vit_lstm | yolo_efficientnet | cnn_bilstm | resnet_attention | yolo_transformer (efficientnet_b0 = reference baseline only)
FOLD = 1                     # 1..10
DATA_ARM = 'without_gan'     # without_gan | with_gan
sh(f"python scripts/colab_preflight.py --fold {FOLD} --data-arm {DATA_ARM} --model {MODEL} --require-cuda")

## 8. INFRASTRUCTURE SMOKE TEST — NOT A PERFORMANCE RESULT

`MODEL` · fold `FOLD` · `DATA_ARM` · 64 training / 32 validation images · 2 epochs (1 head + 1 full) on CUDA. Exercises loading, CLAHE, model, forward/backward, optimiser, validation, metrics, checkpoints and result writing. Written under `results/v2/smoke/` (never mixed with the real matrix). Do not report its metrics as accuracy.

In [ ]:
SMOKE_ID = f'smoke_{MODEL}_fold{FOLD:02d}_{DATA_ARM}'
sh(f"python scripts/run_cv_experiment.py --model {MODEL} --fold {FOLD} --data-arm {DATA_ARM} "
   f"--config configs/cv_v2/smoke.json --require-cuda --smoke --max-train-samples 64 --max-validation-samples 32 "
   f"--out-root results/v2/smoke --experiment-id {SMOKE_ID}")
!cat results/v2/smoke/{SMOKE_ID}/run_summary.json

## 9. Controlled batch — sequential, resumable (`scripts/run_cv_batch.py`)

One cell runs a whole batch through the per-experiment runner: for every (model, fold) it skips COMPLETED runs, re-checks all gates (`colab_preflight.py --model`, and for WITH-GAN `verify_gan_outputs.py --folds F`), trains with `--require-cuda` (resuming from `latest.pt` when one exists), writes `fit_analysis.md/.json`, refreshes the matrix, and writes `results/v2/summaries/<model>_<arm>.md` after each model (`--report` adds the all-models report). A failed gate or run leaves `status.json = FAILED` with the error and the batch continues. After a disconnect: re-run sections 1–6 and this cell again — nothing COMPLETED is ever retrained.

In [ ]:
MODELS = 'cnn_vit_lstm,yolo_efficientnet,cnn_bilstm,resnet_attention,yolo_transformer'   # sequential order
FOLDS = '1-10'
DATA_ARM = 'with_gan'      # with_gan | without_gan
sh(f"python scripts/run_cv_batch.py --models {MODELS} --folds {FOLDS} --data-arm {DATA_ARM} "
   f"--config configs/cv_v2/default.json --require-cuda --report")
!grep -c COMPLETED results/v2/experiment_matrix.csv; grep -c FAILED results/v2/experiment_matrix.csv || true

## 10. After a disconnect

Re-run sections 1–6, then section 9 with `RESUME = True`. `latest.pt` holds model/optimizer/scheduler/scaler/RNG state and the history; the configuration hash must match.